# ANÁLISIS Y ESTANDARIZACIÓN DE PROVEEDORES

In [1]:
import sys
import os
project_root = os.path.dirname(os.getcwd()) 
sys.path.insert(0, project_root)
from utils.imports import *

### VERSIONES DE LIBRERIAS

Python: 3.10.19
pandas: 2.3.3
numpy: 2.2.6
matplotlib: 3.10.0
seaborn: 0.13.2
Pandas: 2.3.3
IPython: 8.38.0


## PRODUCTOS ODOO - INTERFUERZA

In [2]:
notebook_dir = Path.cwd()
raw_path = notebook_dir.parent / 'data' / 'raw'
processed_path = notebook_dir.parent / 'data' / 'processed'
df_odoo = pd.read_csv(raw_path / 'ProductosOdoo.csv')
df_int = pd.read_excel(raw_path / 'ProductosInterfuerza.xlsx')

display(Markdown("### Productos Odoo"))
print(f"Total productos: {len(df_odoo)}")
display(df_odoo.head(5))
display(Markdown("### Productos Interfuerza"))
print(f"Total productos: {len(df_int)}")
display(df_int.head(5))

### Productos Odoo

Total productos: 14215


,id,name,barcode,default_code,x_studio_many2many_field_37q_1irl4uc58,categ_id,seller_ids
0,__export__.product_template_31022_88373f19,ADORNO HALLOWEN,7453066213137,CF00614/autoriza noris,NaN,Cumpleaños / Artículos de Fiesta,NaN
1,__export__.product_template_219315_4256fd8c,ARCHIVADOR VERTICAL 3 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN
2,__export__.product_template_219316_dfced89f,ARCHIVADOR VERTICAL 4 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN
3,__export__.product_template_76612_602c2239,ARTICULOS DE NAVIDAD,93,9,NaN,Navidad / Bolas,NaN
4,__export__.product_template_29991_44f2fe53,ARTICULOS FIESTA,17,1,NaN,Cumpleaños / Artículos de Fiesta,NaN


### Productos Interfuerza

Total productos: 73681


,Id,UPC Code,Ubicacion,Item Number,Tipo,Nombre,Proveedor Principal,Marca,Pais de Origen,Punto de ReOrden,...,Matriz Padre,Matriz Hijo,Arancel,Material,UoM InStock,UoM Compra,UoM Venta,Tags,Peso,Detalle
0,PRO101109,7512155423106,BODEGA PRINCIPAL,A404/6917,PRODUCTO,GORRA FBI 7512155423106,OMER SA,GENERICO,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,SOMBRERO SHERIFF
1,PRO104891,40568000067,NaN,405686/25.95OFERTA5.00,PRODUCTO,VESTIDO LARGO DAMA CHINO,CAPI WORLDWIDE SA,ZZZZZ,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,VESTIDO LARGO DAMA CHINO
2,PRO107818,7465376497664,NaN,9599/HB2473-18,PRODUCTO,BIRRETE GRADUACION NEGRO-AZUL TELA 7465376497664,OMER SA,GENERICO,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,VIRRETE GRADUACION TELA
3,PRO107822,6930180607994,NaN,AFD-799,PRODUCTO,NUMEROS LED CHICO AFD-799,SOLARTE DE PANAMA SA,FIESTAS DAISY,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NUMEROS LED CHICO AFD-799
4,PRO107852,60000001438,BODEGA PRINCIPAL,601430,PRODUCTO,ARREGLO GRADUACION GLOBOS,CORPORACION DAISY SA,DAISY,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,ARREGLO GRADUACION GLOBOS


## ANÁLISIS DE PRODUCTOS SIN PROVEEDOR 

In [3]:
df_odoo_copy, df_int_copy, sellers_diagnostic, sellers_dict, df_int_seller_codes = prepare_data_and_diagnose(
    df_odoo=df_odoo,
    df_int=df_int,
    target_column='seller_ids',
    entity_name='proveedores',
    int_code_col='UPC Code',
    int_entity_col='Proveedor Principal'
)

### Normalización de códigos

#### Muestras de códigos normalizados

,original_odoo,normalizado_odoo,original_int,normalizado_int
0,7453066213137,7453066213137,7512155423106,7512155423106
1,NaN,None,40568000067,40568000067
2,NaN,None,7465376497664,7465376497664
3,93,93,6930180607994,6930180607994
4,17,17,60000001438,60000001438


### Diagnóstico inicial

#### Diagnóstico

Total de productos: 14215
Con proveedores: 8664 (60.95%)
Sin proveedores: 5551 (39.05%)


#### Estadísticas

Total de proveedores con valores únicos: 222
proveedores más frecuente: PROV-ARGELIA INTERNACIONAL S A (851 productos)
proveedores menos frecuente: PATPHA SA (1 productos)
proveedores con un solo producto: 47 (21.17% del total)


##### Creación de diccionario código-proveedores

Mapa creado con 62396 códigos de barras únicos


### Identificación de códigos con múltiples proveedores

In [4]:
multiple_sellers, df_multiple_sellers = analyze_multiple_entities(
    df=df_int_seller_codes,
    code_col='barcode_norm',
    entity_col='Proveedor Principal',
    entity_name='proveedores',
    output_path=raw_path / 'MultiplesProveedores.csv'
)

#### Análisis de múltiples proveedores por código de barras


 ##### Estadísticas generales

Total registros en Interfuerza: 62,512
Total códigos de barras únicos: 62,396
Códigos con múltiples proveedores: 50
Porcentaje del total: 0.08%



 ##### Distribución de multiplicidad

Códigos con 2 proveedores diferentes: 50
Códigos con 3 proveedores diferentes: 0
Códigos con 4+ proveedores diferentes: 0


#### Muestra de códigos con múltiples proveedores

,barcode_norm,Proveedor Principal
0,10100001258,"[DISTRIBUIDORA OFFICE TOTAL, S.A, CENTRO AMERI..."
1,104142,"[INVERSIONES OJENISA SA, CORPORACION DAISY SA]"
2,10514000083,"[SUPRICOM SA, CORPORACION DAISY SA]"
3,105940,"[DESTREZA PANAMA SA, CORPORACION DAISY SA]"
4,105942,"[CORPORACION DAISY SA, MEGA ENSAMBLES INT S.A.]"
5,105944,"[DESTREZA PANAMA SA, LA VICTORIA, S.A. o ALMAC..."
6,105945,"[DESTREZA PANAMA SA, CORPORACION DAISY SA]"
7,105949,"[LA VICTORIA, S.A. o ALMACEN SUN WAH, CORPORAC..."
8,106019,"[DESTREZA PANAMA SA, CORPORACION DAISY SA]"
9,13578105725,"[CARMEL FARMACEUTICA, ABERNATHY S A]"


Archivo 'Proveedores.csv' generado


### Asignación de proveedores a productos sin marca

In [5]:
df_without_seller = df_odoo_copy[sellers_diagnostic].copy()
df_without_seller = df_without_seller[~df_without_seller['barcode_norm'].isin(multiple_sellers.index)]
display(Markdown("#### Análisis de productos sin proveedor"))
print(f"Productos sin marca: {len(df_without_seller)}")

df_without_seller['proveedor_asignado'] = df_without_seller['barcode_norm'].map(sellers_dict)
assigned = df_without_seller['proveedor_asignado'].notna().sum()
total_pendings = len(df_without_seller) - assigned
pendings = df_without_seller[df_without_seller['proveedor_asignado'].isna()]
print(f"✅ Resueltos automáticamente: {assigned} ({assigned/len(df_without_seller)*100:.1f}%)")
print(f"❌ Pendientes (sin coincidencia): {total_pendings} ({total_pendings/len(df_without_seller)*100:.1f}%)\n")

display(Markdown("\n#### MUESTRA DE PRODUCTOS SIN PROVEEDOR PENDIENTES"))
display(pendings)
print("\n")

df_odoo_sellers_import = df_without_seller[df_without_seller['proveedor_asignado'].notna()].copy()
df_odoo_sellers_import = df_odoo_sellers_import[['id', 'proveedor_asignado']]
df_odoo_sellers_import.columns = ['id', 'seller_ids']

df_odoo_sellers_import.to_csv(processed_path / 'ProveedoresParaActualizar.csv', index=False)
display(Markdown("##### Archivo 'ProveedoresParaActualizar.csv' generado"))

#### Análisis de productos sin proveedor

Productos sin marca: 5536
✅ Resueltos automáticamente: 5216 (94.2%)
❌ Pendientes (sin coincidencia): 320 (5.8%)




#### MUESTRA DE PRODUCTOS SIN PROVEEDOR PENDIENTES

,id,name,barcode,default_code,x_studio_many2many_field_37q_1irl4uc58,categ_id,seller_ids,barcode_norm,proveedor_asignado
1,__export__.product_template_219315_4256fd8c,ARCHIVADOR VERTICAL 3 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN,None,NaN
2,__export__.product_template_219316_dfced89f,ARCHIVADOR VERTICAL 4 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN,None,NaN
7,__export__.product_template_219323_100f066b,BATERIA PARA MAQUINA SOPLADORA DE 20 V,NaN,NaN,NaN,All,NaN,None,NaN
9,__export__.product_template_219266_6f6f9bfc,CADENA STRASS SS6 PLATEADO X YDA,109225,CL:SLV-6-SS6,NaN,Sedería / Cintas,NaN,109225,NaN
11,__export__.product_template_219319_c420f23e,CARGADOR PARA MAQUINA SOPLADORA DE 20 V,NaN,NaN,NaN,All,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...
14209,__export__.product_template_218990_b0e13086,[US0510] SILLA EJECUTIVA BLACK,NaN,NaN,NaN,All,NaN,None,NaN
14210,__export__.product_template_220006_59c1bc27,boligrafo,NaN,NaN,NaN,All,NaN,None,NaN
14211,__export__.product_template_220042_92eb42f0,http://www.iamstarmate.com,NaN,NaN,NaN,All,NaN,None,NaN
14212,__export__.product_template_220041_6b35324e,https://facela.com/,NaN,NaN,NaN,All,NaN,None,NaN


##### Archivo 'ProveedoresParaActualizar.csv' generado

### Identificación de proveedores no existentes en Odoo

In [6]:
display(Markdown("#### Análisis de nuevos proveedores"))
assigned_sellers = df_odoo_sellers_import['seller_ids'].astype(str).unique()
print(f"Proveedores únicos por asignar: {len(assigned_sellers)}")

df_odoo_sellers = pd.read_csv(raw_path / 'ProveedoresOdoo.csv')
print(f"Proveedores existentes en Odoo: {len(df_odoo_sellers)}")

# Normalizar nombre de proveedores
def normalize_seller_name(name):
    name = str(name).strip()
    name = re.sub(r'^PROV\s*[-]?\s*', '', name, flags=re.IGNORECASE)     # Eliminar prefijo PROV (con o sin espacios/guiones)
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('ASCII')     # Eliminar tildes
    name = re.sub(r'[^\w\s]', '', name)     # Eliminar caracteres no alfanuméricos
    return ' '.join(name.lower().split())

# Normalizar proveedores existentes (crear mapa: normalizado → original)
odoo_sellers_map = {
    normalize_seller_name(name): name
    for name in df_odoo_sellers['name'].astype(str)
}

# Identificar proveedores nuevos
new_odoo_sellers = []
for seller in assigned_sellers:
    seller_str = str(seller).strip()
    # Limpiar caracteres de lista si viene como "['Prov A', 'Prov B']"
    seller_str = seller_str.strip("[]'\"")
    if normalize_seller_name(seller_str) not in odoo_sellers_map:
        new_odoo_sellers.append(seller_str)

new_odoo_sellers = list(set(new_odoo_sellers))

display(Markdown("##### Muestra de proveedores existentes"))
display(df_odoo_sellers.head())

display(Markdown("##### Muestra de proveedores nuevos"))
print(f"Proveedores nuevos a crear: {len(new_odoo_sellers)}")
print(new_odoo_sellers[:5] if new_odoo_sellers else "Ninguno")

if new_odoo_sellers:
    df_new_odoo_sellers = pd.DataFrame({'name': new_odoo_sellers})
    df_new_odoo_sellers.to_csv(processed_path / 'ProveedoresNuevosOdoo.csv', index=False)
    print("\nArchivo 'ProveedoresNuevosOdoo.csv' generado")

    display(Markdown("#### Muestra de normalización para nombres de proveedores"))
    for seller in new_odoo_sellers[:3]:
        print(f"Original: {seller}")
        print(f"Normalizado: {normalize_seller_name(seller)}\n")
else:
    print("\n No hay proveedores nuevos que crear en Odoo")

#### Análisis de nuevos proveedores

Proveedores únicos por asignar: 210
Proveedores existentes en Odoo: 707


##### Muestra de proveedores existentes

,id,name
0,__export__.res_partner_2277_7ea7be61,PROV-SUSAETA EDICIONES PANAMA SA
1,__export__.res_partner_1827_56fb1616,PROV-DISTEXSA PANAMA S.A.
2,__export__.res_partner_2201_6e8ebc3e,PROV-PRODUCTOS PANAMENOS SA
3,__export__.res_partner_2072_d9fda107,PROV-LELIS SERRANO MARTINEZ
4,__export__.res_partner_2286_66cb0b60,PROV-TEXT BOOK SA


##### Muestra de proveedores nuevos

Proveedores nuevos a crear: 21
["CIA BTESH SA', 'CIA BTESH SA", 'JUAN FRANCISCO GUERRA AGUILAR', 'KEVIN MAN DENG', "MEGA ENSAMBLES INT S.A.', 'MEGA ENSAMBLES INT S.A.", 'SUSAETA EDICIONES PANAMA']

Archivo 'ProveedoresNuevosOdoo.csv' generado


#### Muestra de normalización para nombres de proveedores

Original: CIA BTESH SA', 'CIA BTESH SA
Normalizado: cia btesh sa cia btesh sa

Original: JUAN FRANCISCO GUERRA AGUILAR
Normalizado: juan francisco guerra aguilar

Original: KEVIN MAN DENG
Normalizado: kevin man deng

